In [14]:
from pyspark.sql import SparkSession


spark = (
    SparkSession
        .builder
        .appName("spark-local-app")
        .master("local[*]")
        .getOrCreate()
)

In [15]:
courses_data = [
    {
        "course_id": 101,
        "course_name": "Python Basics",
        "suitable_for": "beginners",
        "enrolment": 3240,
        "stars": 4.7,
        "number_of_rating": 890
    },
    {
        "course_id": 102,
        "course_name": "Data Analysis with Pandas",
        "suitable_for": "intermediate",
        "enrolment": 1870,
        "stars": 4.5,
        "number_of_rating": 520
    },
    {
        "course_id": 103,
        "course_name": "Machine Learning Fundamentals",
        "suitable_for": "intermediate",
        "enrolment": 2150,
        "stars": 4.6,
        "number_of_rating": 610
    },
    {
        "course_id": 104,
        "course_name": "Deep Learning with PyTorch",
        "suitable_for": "advanced",
        "enrolment": 980,
        "stars": 4.8,
        "number_of_rating": 340
    },
    {
        "course_id": 105,
        "course_name": "SQL for Beginners",
        "suitable_for": "beginners",
        "enrolment": 4120,
        "stars": 4.4,
        "number_of_rating": 1120
    },
    {
        "course_id": 106,
        "course_name": "Web Development with React",
        "suitable_for": "intermediate",
        "enrolment": 2730,
        "stars": 4.6,
        "number_of_rating": 780
    },
    {
        "course_id": 107,
        "course_name": "DevOps Essentials",
        "suitable_for": "intermediate",
        "enrolment": 1560,
        "stars": 4.3,
        "number_of_rating": 430
    },
    {
        "course_id": 108,
        "course_name": "Cloud Basics (AWS, Azure, GCP)",
        "suitable_for": "beginners",
        "enrolment": 2990,
        "stars": 4.5,
        "number_of_rating": 670
    },
    {
        "course_id": 109,
        "course_name": "Cybersecurity Fundamentals",
        "suitable_for": "beginners",
        "enrolment": 1940,
        "stars": 4.7,
        "number_of_rating": 510
    },
    {
        "course_id": 110,
        "course_name": "UX/UI Design Principles",
        "suitable_for": "beginners",
        "enrolment": 2310,
        "stars": 4.6,
        "number_of_rating": 690
    }
]


In [16]:
df_courses = spark.createDataFrame(courses_data)

In [17]:
df_courses. \
    sort("suitable_for", "enrolment"). \
    show()

+---------+--------------------+---------+----------------+-----+------------+
|course_id|         course_name|enrolment|number_of_rating|stars|suitable_for|
+---------+--------------------+---------+----------------+-----+------------+
|      104|Deep Learning wit...|      980|             340|  4.8|    advanced|
|      109|Cybersecurity Fun...|     1940|             510|  4.7|   beginners|
|      110|UX/UI Design Prin...|     2310|             690|  4.6|   beginners|
|      108|Cloud Basics (AWS...|     2990|             670|  4.5|   beginners|
|      101|       Python Basics|     3240|             890|  4.7|   beginners|
|      105|   SQL for Beginners|     4120|            1120|  4.4|   beginners|
|      107|   DevOps Essentials|     1560|             430|  4.3|intermediate|
|      102|Data Analysis wit...|     1870|             520|  4.5|intermediate|
|      103|Machine Learning ...|     2150|             610|  4.6|intermediate|
|      106|Web Development w...|     2730|          

In [18]:
from pyspark.sql.functions import desc

df_courses. \
    sort("suitable_for", desc("enrolment")). \
    show()

+---------+--------------------+---------+----------------+-----+------------+
|course_id|         course_name|enrolment|number_of_rating|stars|suitable_for|
+---------+--------------------+---------+----------------+-----+------------+
|      104|Deep Learning wit...|      980|             340|  4.8|    advanced|
|      105|   SQL for Beginners|     4120|            1120|  4.4|   beginners|
|      101|       Python Basics|     3240|             890|  4.7|   beginners|
|      108|Cloud Basics (AWS...|     2990|             670|  4.5|   beginners|
|      110|UX/UI Design Prin...|     2310|             690|  4.6|   beginners|
|      109|Cybersecurity Fun...|     1940|             510|  4.7|   beginners|
|      106|Web Development w...|     2730|             780|  4.6|intermediate|
|      103|Machine Learning ...|     2150|             610|  4.6|intermediate|
|      102|Data Analysis wit...|     1870|             520|  4.5|intermediate|
|      107|   DevOps Essentials|     1560|          

In [20]:
from pyspark.sql.functions import col, when

course_level = when(col("suitable_for") == "beginners", 0).otherwise(when(col("suitable_for") == "intermediate", 1).otherwise(2))

(
    df_courses
        .withColumn("course_level", course_level)
        .orderBy(course_level, col("number_of_rating").desc())
        .show(truncate=False)
)

+---------+------------------------------+---------+----------------+-----+------------+------------+
|course_id|course_name                   |enrolment|number_of_rating|stars|suitable_for|course_level|
+---------+------------------------------+---------+----------------+-----+------------+------------+
|105      |SQL for Beginners             |4120     |1120            |4.4  |beginners   |0           |
|101      |Python Basics                 |3240     |890             |4.7  |beginners   |0           |
|110      |UX/UI Design Principles       |2310     |690             |4.6  |beginners   |0           |
|108      |Cloud Basics (AWS, Azure, GCP)|2990     |670             |4.5  |beginners   |0           |
|109      |Cybersecurity Fundamentals    |1940     |510             |4.7  |beginners   |0           |
|106      |Web Development with React    |2730     |780             |4.6  |intermediate|1           |
|103      |Machine Learning Fundamentals |2150     |610             |4.6  |interme

In [23]:
from pyspark.sql.functions import expr

course_level = expr("""
    CASE
        WHEN suitable_for = 'beginners' THEN 0
        WHEN suitable_for = 'intermediate' THEN 1
        ELSE 2
    END
""")

(
    df_courses
        .withColumn("course_level", course_level)
        .sort(course_level, col("number_of_rating").desc())
        .show()
)

+---------+--------------------+---------+----------------+-----+------------+------------+
|course_id|         course_name|enrolment|number_of_rating|stars|suitable_for|course_level|
+---------+--------------------+---------+----------------+-----+------------+------------+
|      105|   SQL for Beginners|     4120|            1120|  4.4|   beginners|           0|
|      101|       Python Basics|     3240|             890|  4.7|   beginners|           0|
|      110|UX/UI Design Prin...|     2310|             690|  4.6|   beginners|           0|
|      108|Cloud Basics (AWS...|     2990|             670|  4.5|   beginners|           0|
|      109|Cybersecurity Fun...|     1940|             510|  4.7|   beginners|           0|
|      106|Web Development w...|     2730|             780|  4.6|intermediate|           1|
|      103|Machine Learning ...|     2150|             610|  4.6|intermediate|           1|
|      102|Data Analysis wit...|     1870|             520|  4.5|intermediate|  

In [24]:
spark.stop()